In [1]:
import json
import re
import time
import numpy as np
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
# Pasta para os dados brutos coletados
RAW = Path("dados_brutos")
RAW.mkdir(exist_ok=True)

# Pasta para a base tratada (integrada e limpa)
TRATADO = Path("dados_tratados")
TRATADO.mkdir(exist_ok=True)

# Cabeçalho de User-Agent identificando o projeto
HEADERS = {"User-Agent": "UFAM-CienciaDeDados-Trabalho1 (uso academico)"}

PAUSA = 2  # Pausa entre requisições em segundos

# Chave da Google Books API (opcional — a API funciona sem chave, com cota menor)
GOOGLE_BOOKS_API_KEY = None
try:
    from google.colab import userdata
    GOOGLE_BOOKS_API_KEY = userdata.get("GOOGLE_BOOKS_API_KEY")
except Exception:
    import getpass
    GOOGLE_BOOKS_API_KEY = getpass.getpass(
        "Chave da Google Books API (Enter para pular): "
    ) or None

print("Ambiente pronto.")
print("pandas:", pd.__version__)
print("Chave da Google Books API configurada:", bool(GOOGLE_BOOKS_API_KEY))

Chave da Google Books API (Enter para pular): ··········
Ambiente pronto.
pandas: 2.2.3
Chave da Google Books API configurada: True


In [3]:
# Registro de proveniência: para cada fonte, guarda de onde (URL exata),
# quando (data/hora UTC), e como (método e parâmetros) o dado foi coletado.
registro_proveniencia = []

def registrar_proveniencia(fonte, url, metodo, parametros=None, observacao=None):
    entrada = {
        "fonte": fonte,
        "url": url,
        "data_hora_coleta_utc": datetime.now(timezone.utc).isoformat(),
        "metodo": metodo,
        "parametros": parametros or {},
        "observacao": observacao or "",
    }
    registro_proveniencia.append(entrada)
    return entrada

def salvar_proveniencia(caminho=RAW / "proveniencia.json"):
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(registro_proveniencia, f, ensure_ascii=False, indent=2)
    print(f"Proveniência salva em: {caminho} ({len(registro_proveniencia)} entradas)")

**Licença/termos de uso (Wikipédia):** o conteúdo textual da Wikipédia em inglês é distribuído
sob a licença **CC BY-SA 4.0** (Creative Commons Atribuição-CompartilhaIgual), que permite uso,
cópia e redistribuição desde que se dê a devida atribuição à fonte e que
trabalhos derivados sejam compartilhados sob a mesma licença. Neste projeto extraímos apenas dados
factuais das listas (título da obra, ano, autor, título e ano do filme), não o texto integral dos
artigos, o que está plenamente dentro dos termos. A atribuição à Wikipédia está registrada na coluna
`url_lista` e no registro de proveniência.



## Primeira fonte: Wikipédia (scraping) — pares livro→filme

Raspamos as quatro listas alfabéticas de obras de ficção adaptadas para o cinema. Cada linha da
tabela traz duas células: à esquerda `Título (ano), Autor` (a obra literária) e à direita
`Título do filme (ano)`. Extraímos esses campos com expressão regular, preservando o HTML bruto de
cada página coletada.

In [4]:
def raspar(url, user_agent="*"):
    # Verifica no robots.txt do site se a URL pode ser raspada.
    base = re.match(r"^(https?://[^/]+)", url).group(1)
    robots_url = urljoin(base, "/robots.txt")

    # Busca o robots.txt com o mesmo User-Agent das demais requisições
    resp = requests.get(robots_url, headers=HEADERS, timeout=30)
    resp.raise_for_status()

    rp = RobotFileParser()
    rp.set_url(robots_url)
    rp.parse(resp.text.splitlines())

    permitido = rp.can_fetch(user_agent, url)
    print(f"'robots.txt': {robots_url}")
    print(f"Permite acessar {url}? -> {permitido}")
    return permitido, robots_url


# Verificação de robots.txt para a Wikipédia em inglês (fonte raspada)
URL_BASE_WIKI = "https://en.wikipedia.org"
URL_VERIFICACAO = "https://en.wikipedia.org/wiki/List_of_fiction_works_made_into_feature_films"
permitido, robots_url = raspar(URL_VERIFICACAO)

registrar_proveniencia(
    fonte="Wikipedia (En) - Verificação do 'robots.txt'",
    url=robots_url,
    metodo="urllib.robotparser",
    observacao=f"can_fetch para {URL_VERIFICACAO} = {permitido}",
)

assert permitido, "Robots.txt não permite o acesso às listas."

'robots.txt': https://en.wikipedia.org/robots.txt
Permite acessar https://en.wikipedia.org/wiki/List_of_fiction_works_made_into_feature_films? -> True


In [5]:
# WebScraping das listas de obras adaptadas (Wikipédia EN)

DIR_HTML_BRUTO = RAW / "wikipedia_html"
DIR_HTML_BRUTO.mkdir(exist_ok=True)

# As quatro sub-listas alfabéticas de obras de ficção adaptadas para o cinema.
# (a página-índice divide o alfabeto nesses quatro intervalos)
PAGINAS_LISTA = [
    "List of fiction works made into feature films (0–9, A–C)",
    "List of fiction works made into feature films (D–J)",
    "List of fiction works made into feature films (K–R)",
    "List of fiction works made into feature films (S–Z)",
]


def baixar_lista(titulo_pagina, indice_pagina):
    # Baixa uma página de lista, salva o HTML bruto e devolve (html, url).
    url = URL_BASE_WIKI + "/wiki/" + titulo_pagina.replace(" ", "_")
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()

    caminho_html = DIR_HTML_BRUTO / f"lista_pagina_{indice_pagina:02d}.html"
    caminho_html.write_text(resp.text, encoding="utf-8")

    registrar_proveniencia(
        fonte="Wikipedia (En) - List of fiction works made into feature films",
        url=resp.url,
        metodo="requests.get + pandas.read_html",
        parametros={"pagina": indice_pagina, "status_http": resp.status_code},
        observacao=f"HTML bruto salvo em {caminho_html}",
    )
    return resp.text, resp.url


def separar_obra(texto):
    # Da célula esquerda "Título (ano), Autor" extrai (titulo, ano, autor).
    # Usa o ÚLTIMO "(ano)" de 4 dígitos como o ano da obra (há títulos com
    # parênteses no meio, ex.: "The Dancing Girl (舞姫, Maihime) (1951), Kawabata").
    if not isinstance(texto, str):
        return None, None, None
    texto = texto.strip()
    m = re.search(r"\((\d{4})\)", texto)
    ano = int(m.group(1)) if m else None
    # título = tudo antes do último "(ano)"; autor = o que vem depois da vírgula final
    if m:
        titulo = texto[: m.start()].strip().rstrip(",").strip()
        resto = texto[m.end():].strip()
        autor = resto.lstrip(",").strip() or None
    else:
        titulo, autor = texto, None
    return titulo or None, ano, autor


def separar_filme(texto):
    # Da célula direita "Título do filme (ano)" extrai (titulo_filme, ano_filme).
    if not isinstance(texto, str):
        return None, None
    texto = texto.strip()
    m = re.search(r"\((\d{4})\)", texto)
    ano = int(m.group(1)) if m else None
    titulo = texto[: m.start()].strip() if m else texto
    return (titulo or None), ano


# Loop de coleta com pausa entre requisições (não sobrecarrega o servidor)
linhas = []
for i, titulo_pagina in enumerate(PAGINAS_LISTA, start=1):
    print(f"Coletando lista {i}/{len(PAGINAS_LISTA)}: {titulo_pagina}")
    try:
        html, url_pagina = baixar_lista(titulo_pagina, i)
    except requests.HTTPError as e:
        print(f"  ATENÇÃO: falha ao baixar ({e}) — pulando esta página.")
        continue

    # pandas.read_html extrai todas as tabelas da página
    tabelas = pd.read_html(html)
    n_antes = len(linhas)
    for tab in tabelas:
        # só as tabelas de duas colunas no formato obra | filme
        if tab.shape[1] < 2:
            continue
        col_obra, col_filme = tab.columns[0], tab.columns[1]
        # confirma que é a tabela certa pelo cabeçalho
        if "fiction" not in str(col_obra).lower() and "work" not in str(col_obra).lower():
            continue
        for _, linha in tab.iterrows():
            titulo_livro, ano_livro, autor = separar_obra(linha[col_obra])
            titulo_filme, ano_filme = separar_filme(linha[col_filme])
            if not titulo_livro:
                continue
            linhas.append({
                "titulo_livro_wiki": titulo_livro,
                "ano_publicacao_livro_wiki": ano_livro,
                "autor_wiki": autor,
                "titulo_filme_wiki": titulo_filme,
                "ano_filme_wiki": ano_filme,
                "url_lista": url_pagina,
            })
    print(f"  -> {len(linhas) - n_antes} pares extraídos")
    if i < len(PAGINAS_LISTA):
        time.sleep(PAUSA)

print(f"\nTotal de pares livro→filme coletados: {len(linhas)}")

Coletando lista 1/4: List of fiction works made into feature films (0–9, A–C)


/tmp/ipykernel_815/2195690449.py:76: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tabelas = pd.read_html(html)


  -> 769 pares extraídos
Coletando lista 2/4: List of fiction works made into feature films (D–J)


/tmp/ipykernel_815/2195690449.py:76: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tabelas = pd.read_html(html)


  -> 1208 pares extraídos
Coletando lista 3/4: List of fiction works made into feature films (K–R)


/tmp/ipykernel_815/2195690449.py:76: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tabelas = pd.read_html(html)


  -> 986 pares extraídos
Coletando lista 4/4: List of fiction works made into feature films (S–Z)
  -> 858 pares extraídos

Total de pares livro→filme coletados: 3821


/tmp/ipykernel_815/2195690449.py:76: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tabelas = pd.read_html(html)


In [6]:
# DataFrame bruto: exatamente como veio do parsing, sem limpeza de conteúdo
df_wikipedia_bruto = pd.DataFrame(linhas)

print("Dimensões:", df_wikipedia_bruto.shape)
print("\nExemplos:")
display(df_wikipedia_bruto.head(10))

# Salva o CSV bruto (o HTML de cada página já foi salvo em wikipedia_html/)
caminho_csv_bruto = RAW / "wikipedia_listas_pares.csv"
df_wikipedia_bruto.to_csv(caminho_csv_bruto, index=False, encoding="utf-8")

salvar_proveniencia()
print(f"\nSalvo: {caminho_csv_bruto}")

Dimensões: (3821, 6)

Exemplos:


,titulo_livro_wiki,ano_publicacao_livro_wiki,autor_wiki,titulo_filme_wiki,ano_filme_wiki,url_lista
0,The 25th Hour,2001.0,David Benioff,25th Hour,2002.0,https://en.wikipedia.org/wiki/List_of_fiction_...
1,"3 Assassins (グラスホッパー, Gurasuhoppā)",2004.0,Kōtarō Isaka,Grasshopper,2015.0,https://en.wikipedia.org/wiki/List_of_fiction_...
2,4.50 from Paddington,1957.0,Agatha Christie,"Murder, She Said",1961.0,https://en.wikipedia.org/wiki/List_of_fiction_...
3,4.50 from Paddington,1957.0,Agatha Christie,Crime Is Our Business (French: Le Crime est no...,2008.0,https://en.wikipedia.org/wiki/List_of_fiction_...
4,58 Minutes,1987.0,Walter Wager,Die Hard 2,1990.0,https://en.wikipedia.org/wiki/List_of_fiction_...
5,"69 (シクスティナイン, Shikusutinain)",1987.0,Ryu Murakami,69,2004.0,https://en.wikipedia.org/wiki/List_of_fiction_...
6,The A.B.C. Murders,1936.0,Agatha Christie,The Alphabet Murders,1966.0,https://en.wikipedia.org/wiki/List_of_fiction_...
7,À ton image,1998.0,Louise L. Lambrichs,À ton image,2004.0,https://en.wikipedia.org/wiki/List_of_fiction_...
8,About a Boy,1998.0,Nick Hornby,About a Boy,2002.0,https://en.wikipedia.org/wiki/List_of_fiction_...
9,About Schmidt,1996.0,Louis Begley,About Schmidt,2002.0,https://en.wikipedia.org/wiki/List_of_fiction_...


Proveniência salva em: dados_brutos/proveniencia.json (5 entradas)

Salvo: dados_brutos/wikipedia_listas_pares.csv


In [7]:
# Diagnóstico de volume por ano do filme
# Serve para escolher um limiar temporal que reduza o número de chamadas às APIs
# sem sacrificar dados úteis. Filmes mais recentes tendem a ter mais informação
# preenchida na TMDB (orçamento, receita, votos).

anos = pd.to_numeric(df_wikipedia_bruto["ano_filme_wiki"], errors="coerce")

print(f"Total de pares coletados:            {len(df_wikipedia_bruto)}")
print(f"Pares com ano de filme identificado: {anos.notna().sum()}")
print(f"Pares sem ano de filme:              {anos.isna().sum()}\n")

# Quantos sobram para vários limiares candidatos
print("Pares restantes por limiar (ano do filme >=):")
for corte in [1950, 1960, 1970, 1980, 1990, 2000, 2010]:
    n = (anos >= corte).sum()
    print(f"  a partir de {corte}: {n:5d} pares")

# Distribuição por década, para enxergar onde estão os dados
print("\nDistribuição por década do filme:")
decadas = (anos // 10 * 10).dropna().astype(int)
print(decadas.value_counts().sort_index().to_string())

# Foco no seu candidato (>= 1970)
n70 = (anos >= 1970).sum()
print(f"\n>>> Com limiar 1970: {n70} pares "
      f"(~{n70*4} chamadas de API no total, contando GB + TMDB busca + TMDB detalhes + keywords)")

Total de pares coletados:            3821
Pares com ano de filme identificado: 3772
Pares sem ano de filme:              49

Pares restantes por limiar (ano do filme >=):
  a partir de 1950:  3005 pares
  a partir de 1960:  2650 pares
  a partir de 1970:  2265 pares
  a partir de 1980:  1770 pares
  a partir de 1990:  1347 pares
  a partir de 2000:   911 pares
  a partir de 2010:   406 pares

Distribuição por década do filme:
ano_filme_wiki
1900     12
1910    114
1920    158
1930    244
1940    239
1950    355
1960    385
1970    495
1980    423
1990    436
2000    505
2010    264
2020    142

>>> Com limiar 1970: 2265 pares (~9060 chamadas de API no total, contando GB + TMDB busca + TMDB detalhes + keywords)


### Recorte temporal: filmes a partir de 2000

O scraping rendeu 3821 pares livro→filme. Utilizar todos exigiria 15 mil chamadas de API, o que tornaria a coleta irreprodutível na prática. Aplicamos então um
**recorte temporal**: mantemos apenas filmes lançados **a partir do ano 2000** (911 pares).

A escolha não é só de volume, filmes recentes têm cobertura muito melhor de dados na TMDB
(orçamento, receita, votos), justamente os campos que mais sofriam com ausências na versão anterior
da base. O recorte é **determinístico** (não é amostra aleatória) e está documentado como limitação
de cobertura no dataset card: a base passa a representar *adaptações recentes*, não adaptações em
geral.

In [8]:
# Recorte: filmes a partir de 2000, e apenas pares com ano de filme identificado.
# Trabalhamos sobre uma cópia; df_wikipedia_bruto (todos os 3821) segue intocado.
ANO_CORTE = 2000

_anos = pd.to_numeric(df_wikipedia_bruto["ano_filme_wiki"], errors="coerce")
df_wikipedia = df_wikipedia_bruto[_anos >= ANO_CORTE].copy().reset_index(drop=True)

print(f"Pares antes do recorte: {len(df_wikipedia_bruto)}")
print(f"Pares após recorte (filme >= {ANO_CORTE}): {len(df_wikipedia)}")
print(f"Descartados (anteriores a {ANO_CORTE} ou sem ano): "
      f"{len(df_wikipedia_bruto) - len(df_wikipedia)}")
display(df_wikipedia.head())

Pares antes do recorte: 3821
Pares após recorte (filme >= 2000): 911
Descartados (anteriores a 2000 ou sem ano): 2910


,titulo_livro_wiki,ano_publicacao_livro_wiki,autor_wiki,titulo_filme_wiki,ano_filme_wiki,url_lista
0,The 25th Hour,2001.0,David Benioff,25th Hour,2002.0,https://en.wikipedia.org/wiki/List_of_fiction_...
1,"3 Assassins (グラスホッパー, Gurasuhoppā)",2004.0,Kōtarō Isaka,Grasshopper,2015.0,https://en.wikipedia.org/wiki/List_of_fiction_...
2,4.50 from Paddington,1957.0,Agatha Christie,Crime Is Our Business (French: Le Crime est no...,2008.0,https://en.wikipedia.org/wiki/List_of_fiction_...
3,"69 (シクスティナイン, Shikusutinain)",1987.0,Ryu Murakami,69,2004.0,https://en.wikipedia.org/wiki/List_of_fiction_...
4,À ton image,1998.0,Louise L. Lambrichs,À ton image,2004.0,https://en.wikipedia.org/wiki/List_of_fiction_...


**Licença/termos de uso (Google Books API):** o uso da API é regido pelos
[Termos de Serviço das APIs do Google](https://developers.google.com/terms) e pela política da Books
API, que permite consultas para pesquisa e uso acadêmico sem custo, dentro dos limites de cota. É
proibido usar os dados para criar um serviço concorrente ao Google Books ou redistribuir os dados
brutos como produto próprio. Aqui usamos apenas metadados agregados (**nota média** e **número de
avaliações** do livro), sem redistribuir a API.



In [9]:
# API Google Books — enriquece cada par com a avaliação do livro (nota e nº de avaliações).
GOOGLEBOOKS_BRUTO = RAW / "google_books_json"
GOOGLEBOOKS_BRUTO.mkdir(exist_ok=True)

GOOGLE_BOOKS_ENDPOINT = "https://www.googleapis.com/books/v1/volumes"


def consultar_livro(titulo, autor, indice, max_tentativas=3):
    # Consulta a Google Books API por título + autor e salva o JSON bruto da resposta.
    # Usar o autor (vindo da Wikipédia) reduz muito o risco de casar com outro livro.
    q = f'intitle:{titulo}'
    if isinstance(autor, str) and autor.strip():
        q += f' inauthor:{autor}'
    params = {"q": q, "langRestrict": "en", "maxResults": 5}
    if GOOGLE_BOOKS_API_KEY:
        params["key"] = GOOGLE_BOOKS_API_KEY

    for tentativa in range(1, max_tentativas + 1):
        resp = requests.get(GOOGLE_BOOKS_ENDPOINT, params=params, headers=HEADERS, timeout=30)
        if resp.status_code in (429, 503):
            print(f"  {resp.status_code} para '{titulo}', tentativa {tentativa}. Aguardando...")
            time.sleep(PAUSA * 5)
            continue
        resp.raise_for_status()
        break
    else:
        raise RuntimeError(f"Falha ao consultar '{titulo}' após {max_tentativas} tentativas.")

    caminho_json = GOOGLEBOOKS_BRUTO / f"googlebooks_{indice:04d}.json"
    caminho_json.write_text(resp.text, encoding="utf-8")

    registrar_proveniencia(
        fonte="Google Books API",
        url=resp.url,
        metodo="requests.get (Google Books API v1/volumes)",
        parametros={"q": q, "status_http": resp.status_code},
        observacao=f"JSON bruto salvo em {caminho_json}",
    )
    return resp.json()


def extrair_info_livro(payload, titulo_livro_wiki, autor_wiki):
    # Extrai os campos de interesse do primeiro volume retornado.
    itens = payload.get("items")
    base = {
        "titulo_livro_wiki": titulo_livro_wiki,
        "titulo_google_books": None,
        "autor_google_books": None,
        "nota_media": None,
        "n_avaliacoes": None,
        "categorias": None,
        "idioma": None,
        "editora": None,
    }
    if not itens:
        return base
    info = itens[0].get("volumeInfo", {})
    base.update({
        "titulo_google_books": info.get("title"),
        "autor_google_books": ", ".join(info.get("authors", [])) or None,
        "nota_media": info.get("averageRating"),
        "n_avaliacoes": info.get("ratingsCount"),
        "categorias": ", ".join(info.get("categories", [])) or None,
        "idioma": info.get("language"),
        "editora": info.get("publisher"),
    })
    return base

In [10]:
# Limpeza do título do livro: remove trechos de tradução/idioma tipo "(German: ...)"
# e parênteses residuais que não sejam o ano.
def limpar_titulo_livro(titulo):
    if not isinstance(titulo, str):
        return titulo
    # remove parênteses que contenham ':' (traduções) ou nomes de idioma
    t = re.sub(r"\s*\((?:[A-Z][a-z]+:\s*)?[^)]*\)", "", titulo)
    return re.sub(r"\s+", " ", t).strip()

df_wikipedia["titulo_livro_limpo"] = df_wikipedia["titulo_livro_wiki"].apply(limpar_titulo_livro)

# Livros ÚNICOS (título limpo + autor) — o Google Books só precisa de um por livro.
livros_unicos = (
    df_wikipedia[["titulo_livro_limpo", "autor_wiki"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f"Pares (linhas): {len(df_wikipedia)}")
print(f"Livros únicos a consultar no Google Books: {len(livros_unicos)}")

Pares (linhas): 911
Livros únicos a consultar no Google Books: 522


In [11]:
# Flag de série
df_wikipedia["eh_serie"] = df_wikipedia["titulo_livro_wiki"].str.contains(
    r"\(series\)|\(\d{4}\s*[–-]\s*\d{4}\)", case=False, na=False, regex=True
)
print("Linhas marcadas como série:", int(df_wikipedia["eh_serie"].sum()))

Linhas marcadas como série: 117


In [12]:
resultados_livros = []
for i, linha in livros_unicos.iterrows():
    idx = i + 1
    titulo = linha["titulo_livro_limpo"]
    autor = linha["autor_wiki"]
    print(f"[{idx}/{len(livros_unicos)}] Google Books: {titulo}")
    payload = consultar_livro(titulo, autor, idx)
    info = extrair_info_livro(payload, titulo, autor)
    info["autor_wiki"] = autor          # guarda a chave de merge
    resultados_livros.append(info)
    time.sleep(PAUSA)

df_google_books_bruto = pd.DataFrame(resultados_livros)
print("\nDimensões:", df_google_books_bruto.shape)

[1/522] Google Books: The 25th Hour
[2/522] Google Books: 3 Assassins
[3/522] Google Books: 4.50 from Paddington
[4/522] Google Books: 69
[5/522] Google Books: À ton image
[6/522] Google Books: About a Boy
[7/522] Google Books: About Schmidt
[8/522] Google Books: Adam Resurrected
[9/522] Google Books: Adams Fall
[10/522] Google Books: Adiliya by the River
[11/522] Google Books: Adolphe
[12/522] Google Books: The Adventures of Captain Alatriste, Arturo Pérez-Reverte
[13/522] Google Books: Adventures of Huckleberry Finn
[14/522] Google Books: The Adventures of Maya the Bee
[15/522] Google Books: The Adventures of Pinocchio
[16/522] Google Books: The Adventures of Tom Sawyer
[17/522] Google Books: After
[18/522] Google Books: After Ever Happy
[19/522] Google Books: After Everything
[20/522] Google Books: After We Collided
[21/522] Google Books: After We Fell
[22/522] Google Books: Airborn
[23/522] Google Books: All Quiet on the Western Front
[24/522] Google Books: All She Was Worth
[25/52

## Terceira fonte: TMDB API (dados dos filmes)

Até aqui temos, por par:
- **Wikipédia (scraping):** título do livro, ano da 1ª edição, autor, título e ano do filme.
- **Google Books API:** avaliação dos leitores sobre o **livro** (`nota_media`, `n_avaliacoes`).

Falta o lado do **filme**: bilheteria, orçamento, nota do público, popularidade, duração, gêneros.
Isso vem da **TMDB (The Movie Database) API**. A busca usa o **título do filme + ano** (vindos da
Wikipédia) e restringe o idioma a inglês, o que torna o casamento muito mais preciso do que a versão
anterior. Depois dos detalhes, consultamos também as **keywords** de cada filme para confirmar, via
`based on novel or book` (keyword 818), que ele é de fato uma adaptação de livro.

**Licença/termos de uso:** a TMDB exige a atribuição *"This product uses the TMDB API but is not
endorsed or certified by TMDB"* e proíbe uso que viole seus Termos de Serviço. Uso acadêmico é
permitido.

In [13]:
# Configuração da TMDB API
TMDB_TOKEN = None
try:
    from google.colab import userdata
    TMDB_TOKEN = userdata.get("TMDB_TOKEN")
except Exception:
    import getpass
    TMDB_TOKEN = getpass.getpass("Token (Bearer v4) da TMDB API (Enter para pular): ") or None

TMDB_ENDPOINT_SEARCH = "https://api.themoviedb.org/3/search/movie"
TMDB_ENDPOINT_DETAILS = "https://api.themoviedb.org/3/movie/{id}"
TMDB_ENDPOINT_KEYWORDS = "https://api.themoviedb.org/3/movie/{id}/keywords"

TMDB_HEADERS = dict(HEADERS)  # reaproveita o User-Agent
if TMDB_TOKEN:
    TMDB_HEADERS["Authorization"] = f"Bearer {TMDB_TOKEN}"

TMDB_BRUTO = RAW / "tmdb_json"
TMDB_BRUTO.mkdir(exist_ok=True)

print("Chave da TMDB API configurada:", bool(TMDB_TOKEN))

Token (Bearer v4) da TMDB API (Enter para pular): ··········
Chave da TMDB API configurada: True


In [14]:
def _get_tmdb(url, params, descricao, max_tentativas=3):
    # GET genérico com retry para erros de cota/conexão.
    for tentativa in range(1, max_tentativas + 1):
        try:
            resp = requests.get(url, params=params, headers=TMDB_HEADERS, timeout=30)
        except requests.exceptions.ConnectionError as e:
            print(f"  ConnectionError em {descricao}, tentativa {tentativa}: aguardando...")
            time.sleep(PAUSA * 5)
            continue
        if resp.status_code == 429:
            espera = int(resp.headers.get("Retry-After", PAUSA * 5))
            print(f"  429 em {descricao}, tentativa {tentativa}. Aguardando {espera}s...")
            time.sleep(espera)
            continue
        resp.raise_for_status()
        return resp
    raise RuntimeError(f"Falha em {descricao} após {max_tentativas} tentativas.")


def buscar_filme_tmdb(titulo, ano, indice):
    # Busca o filme por título + ano (primary_release_year desempata homônimos).
    # Fallback (b): se a busca COM ano não retorna nada, tenta de novo SEM ano,
    # para não perder filmes cujo ano na Wikipédia difere ~1 do ano na TMDB.
    def _buscar(com_ano):
        params = {"query": titulo, "language": "en-US", "include_adult": "false"}
        if com_ano and pd.notna(ano):
            params["primary_release_year"] = int(ano)
        resp = _get_tmdb(TMDB_ENDPOINT_SEARCH, params, f"busca '{titulo}'")
        return resp

    # 1ª tentativa: com ano
    resp = _buscar(com_ano=True)
    resultados = resp.json().get("results", [])
    usou_ano = True

    # fallback: sem ano, se nada veio (e havia um ano para tentar)
    if not resultados and pd.notna(ano):
        resp = _buscar(com_ano=False)
        resultados = resp.json().get("results", [])
        usou_ano = False

    # salva o JSON bruto da busca que efetivamente usamos
    (TMDB_BRUTO / f"tmdb_busca_{indice:04d}.json").write_text(resp.text, encoding="utf-8")
    registrar_proveniencia(
        fonte="TMDB API - busca por filme", url=resp.url,
        metodo="requests.get (TMDB /search/movie)",
        parametros={"query": titulo, "ano": None if pd.isna(ano) else int(ano),
                    "usou_ano": usou_ano, "status_http": resp.status_code},
        observacao=f"JSON salvo em tmdb_busca_{indice:04d}.json"
                   f"{' (fallback sem ano)' if not usou_ano else ''}",
    )
    return resultados[0] if resultados else None

def detalhar_filme_tmdb(tmdb_id, indice):
    url = TMDB_ENDPOINT_DETAILS.format(id=tmdb_id)
    resp = _get_tmdb(url, {"language": "en-US"}, f"detalhes id={tmdb_id}")
    (TMDB_BRUTO / f"tmdb_detalhes_{indice:04d}.json").write_text(resp.text, encoding="utf-8")
    registrar_proveniencia(
        fonte="TMDB API - detalhes do filme", url=resp.url,
        metodo="requests.get (TMDB /movie/{id})",
        parametros={"tmdb_id": tmdb_id, "status_http": resp.status_code},
        observacao=f"JSON salvo em tmdb_detalhes_{indice:04d}.json",
    )
    return resp.json()


def keywords_filme_tmdb(tmdb_id, indice):
    # Consulta as keywords do filme; usamos a 818 "based on novel or book" para validar.
    url = TMDB_ENDPOINT_KEYWORDS.format(id=tmdb_id)
    resp = _get_tmdb(url, {}, f"keywords id={tmdb_id}")
    (TMDB_BRUTO / f"tmdb_keywords_{indice:04d}.json").write_text(resp.text, encoding="utf-8")
    registrar_proveniencia(
        fonte="TMDB API - keywords do filme", url=resp.url,
        metodo="requests.get (TMDB /movie/{id}/keywords)",
        parametros={"tmdb_id": tmdb_id, "status_http": resp.status_code},
        observacao=f"JSON salvo em tmdb_keywords_{indice:04d}.json",
    )
    kws = resp.json().get("keywords", [])
    nomes = [k.get("name", "").lower() for k in kws]
    eh_adaptacao = any(("novel" in n or "book" in n) for n in nomes)
    return nomes, eh_adaptacao

In [15]:
# Loop de coleta na TMDB: para cada par, busca o filme (título+ano), detalha e
# consulta as keywords.
resultados_tmdb = []

for i, linha in df_wikipedia.reset_index(drop=True).iterrows():
    idx = i + 1
    titulo_filme = linha["titulo_filme_wiki"]
    ano_filme = linha["ano_filme_wiki"]
    print(f"[{idx}/{len(df_wikipedia)}] TMDB: {titulo_filme} ({ano_filme})")

    registro = {
        "titulo_filme_wiki": titulo_filme,
        "ano_filme_wiki": ano_filme,
        "tmdb_id": None, "titulo_tmdb": None, "data_lancamento": None,
        "idioma_original": None, "generos": None, "duracao_min": None,
        "orcamento": None, "receita": None, "popularidade": None,
        "media_votos": None, "contagem_votos": None,
        "keywords_tmdb": None, "eh_adaptacao_livro": None,
    }

    candidato = buscar_filme_tmdb(titulo_filme, ano_filme, idx)
    if candidato is None:
        print("  -> nenhum resultado")
        resultados_tmdb.append(registro)
        time.sleep(PAUSA)
        continue

    tmdb_id = candidato.get("id")
    registro["tmdb_id"] = tmdb_id
    registro["titulo_tmdb"] = candidato.get("title")

    detalhes = detalhar_filme_tmdb(tmdb_id, idx)
    registro.update({
        "data_lancamento": detalhes.get("release_date") or None,
        "idioma_original": detalhes.get("original_language"),
        "generos": ", ".join(g["name"] for g in detalhes.get("genres", [])) or None,
        "duracao_min": detalhes.get("runtime"),
        "orcamento": detalhes.get("budget"),
        "receita": detalhes.get("revenue"),
        "popularidade": detalhes.get("popularity"),
        "media_votos": detalhes.get("vote_average"),
        "contagem_votos": detalhes.get("vote_count"),
    })
    time.sleep(PAUSA)

    nomes_kw, eh_adaptacao = keywords_filme_tmdb(tmdb_id, idx)
    registro["keywords_tmdb"] = ", ".join(nomes_kw) or None
    registro["eh_adaptacao_livro"] = eh_adaptacao

    resultados_tmdb.append(registro)
    time.sleep(PAUSA)

df_tmdb_bruto = pd.DataFrame(resultados_tmdb)
print("\nDimensões:", df_tmdb_bruto.shape)
display(df_tmdb_bruto.head())

caminho_csv_tmdb = RAW / "tmdb_resultados.csv"
df_tmdb_bruto.to_csv(caminho_csv_tmdb, index=False, encoding="utf-8")
salvar_proveniencia()
print(f"Salvo: {caminho_csv_tmdb}")

[1/911] TMDB: 25th Hour (2002.0)
[2/911] TMDB: Grasshopper (2015.0)
[3/911] TMDB: Crime Is Our Business (French: Le Crime est notre affaire) (2008.0)
  -> nenhum resultado
[4/911] TMDB: 69 (2004.0)
[5/911] TMDB: À ton image (2004.0)
[6/911] TMDB: About a Boy (2002.0)
[7/911] TMDB: About Schmidt (2002.0)
[8/911] TMDB: Adam Resurrected (2009.0)
[9/911] TMDB: Abandon (2002.0)
[10/911] TMDB: Green Tea (2003.0)
[11/911] TMDB: Adolphe (2002.0)
[12/911] TMDB: Alatriste (2006.0)
[13/911] TMDB: The Adventures of Huck Finn (German: Die Abenteuer des Huck Finn) (2012.0)
  -> nenhum resultado
[14/911] TMDB: Maya the Bee (2015.0)
[15/911] TMDB: Maya the Bee: The Honey Games (2018.0)
[16/911] TMDB: Maya the Bee: The Golden Orb (2021.0)
[17/911] TMDB: Geppetto (2000.0)
[18/911] TMDB: Pinocchio (2002.0)
[19/911] TMDB: Pinocchio 3000 (2004.0)
[20/911] TMDB: Welcome Back Pinocchio (Italian: Bentornato Pinocchio) (2007.0)
  -> nenhum resultado
[21/911] TMDB: Pinocchio (2008.0)
[22/911] TMDB: Pistachio – 

,titulo_filme_wiki,ano_filme_wiki,tmdb_id,titulo_tmdb,data_lancamento,idioma_original,generos,duracao_min,orcamento,receita,popularidade,media_votos,contagem_votos,keywords_tmdb,eh_adaptacao_livro
0,25th Hour,2002.0,1429.0,25th Hour,2002-12-19,en,"Crime, Drama",135.0,5000000.0,23932055.0,7.3098,7.312,2647.0,"drug dealer, new york city, friendship, dreams...",True
1,Grasshopper,2015.0,313219.0,Grasshopper,2015-11-07,ja,"Crime, Thriller, Action, Drama",119.0,0.0,0.0,1.3858,5.900,15.0,based on novel or book,True
2,Crime Is Our Business (French: Le Crime est no...,2008.0,NaN,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,None,None
3,69,2004.0,119776.0,69,2004-07-10,ja,"Drama, Comedy",114.0,0.0,0.0,2.2873,6.500,15.0,japanese economic miracle,False
4,À ton image,2004.0,47109.0,In Your Image,2004-05-26,fr,"Drama, Science Fiction, Thriller",94.0,0.0,0.0,1.7590,4.833,18.0,woman director,False


Proveniência salva em: dados_brutos/proveniencia.json (3132 entradas)
Salvo: dados_brutos/tmdb_resultados.csv


In [16]:
# Profiling da TMDB recém-coletada: taxa de match e cobertura da validação por keyword.
print("Dimensões:", df_tmdb_bruto.shape)

taxa_match = df_tmdb_bruto["tmdb_id"].notna().mean() * 100
print(f"Taxa de match TMDB (achou filme): {taxa_match:.1f}%")

# Entre os que casaram, quantos a TMDB confirma como adaptação de livro (keyword 818 & cia.)
com_match = df_tmdb_bruto[df_tmdb_bruto["tmdb_id"].notna()]
if len(com_match):
    taxa_adap = com_match["eh_adaptacao_livro"].mean() * 100
    print(f"Entre os que casaram, confirmados como adaptação de livro: {taxa_adap:.1f}%")

print("\nAusentes por coluna:")
faltantes = df_tmdb_bruto.isnull().sum()
print(faltantes[faltantes > 0].sort_values(ascending=False))

Dimensões: (911, 15)
Taxa de match TMDB (achou filme): 93.0%
Entre os que casaram, confirmados como adaptação de livro: 48.5%

Ausentes por coluna:
keywords_tmdb         161
generos                66
data_lancamento        65
titulo_tmdb            64
tmdb_id                64
idioma_original        64
duracao_min            64
receita                64
orcamento              64
popularidade           64
media_votos            64
contagem_votos         64
eh_adaptacao_livro     64
dtype: int64


In [17]:
# INSPEÇÃO: quais filmes NÃO casaram na TMDB?
sem_match = df_tmdb_bruto[df_tmdb_bruto["tmdb_id"].isna()]
print(f"Filmes sem match na TMDB: {len(sem_match)}\n")


display(sem_match[["titulo_filme_wiki", "ano_filme_wiki"]].reset_index(drop=True))

Filmes sem match na TMDB: 64



,titulo_filme_wiki,ano_filme_wiki
0,Crime Is Our Business (French: Le Crime est no...,2008.0
1,The Adventures of Huck Finn (German: Die Abent...,2012.0
2,Welcome Back Pinocchio (Italian: Bentornato Pi...,2007.0
3,Tom and Huck (German: Tom und Hacke),2012.0
4,"Helpless (Korean: 화차, romanized: Hwacha)",2012.0
...,...,...
59,Treasure Island (German: L'Île aux trésors,2007.0
60,Treasure Island (German: Die Schatzinsel),2007.0
61,Troublesome Night 16 (Chinese: 水滸傳),2002.0
62,White Fang (French: Croc-Blanc),2018.0


In [18]:
# Recuperação dos filmes sem match: o título do filme veio com trechos de tradução
# grudados, o que fez a busca na TMDB falhar.
# Aqui limpamos o título do filme e reconsultamos SÓ os 64 sem match.

def limpar_titulo_filme(titulo):
    if not isinstance(titulo, str):
        return titulo
    # remove parênteses de tradução/idioma (contêm ':' ou nome de idioma) e romanização
    t = re.sub(r"\s*\((?:[A-Z][a-z]+:\s*)?[^)]*\)", "", titulo)
    return re.sub(r"\s+", " ", t).strip()

sem_match_idx = df_tmdb_bruto.index[df_tmdb_bruto["tmdb_id"].isna()].tolist()
print(f"Reconsultando {len(sem_match_idx)} filmes com título limpo...\n")

_offset = 5000  # índice alto para não sobrescrever os JSONs já salvos
recuperados = 0

for pos, idx_df in enumerate(sem_match_idx, start=1):
    titulo_cru = df_tmdb_bruto.at[idx_df, "titulo_filme_wiki"]
    ano = df_tmdb_bruto.at[idx_df, "ano_filme_wiki"]
    titulo_limpo = limpar_titulo_filme(titulo_cru)
    if titulo_limpo == titulo_cru:
        continue  # nada mudou, não adianta reconsultar
    print(f"[{pos}/{len(sem_match_idx)}] {titulo_limpo} ({ano})")

    candidato = buscar_filme_tmdb(titulo_limpo, ano, _offset + pos)
    if candidato is None:
        continue

    tmdb_id = candidato.get("id")
    detalhes = detalhar_filme_tmdb(tmdb_id, _offset + pos)
    nomes_kw, eh_adap = keywords_filme_tmdb(tmdb_id, _offset + pos)

    # preenche a linha existente no df_tmdb_bruto (não cria linha nova)
    df_tmdb_bruto.at[idx_df, "tmdb_id"] = tmdb_id
    df_tmdb_bruto.at[idx_df, "titulo_tmdb"] = candidato.get("title")
    df_tmdb_bruto.at[idx_df, "data_lancamento"] = detalhes.get("release_date") or None
    df_tmdb_bruto.at[idx_df, "idioma_original"] = detalhes.get("original_language")
    df_tmdb_bruto.at[idx_df, "generos"] = ", ".join(g["name"] for g in detalhes.get("genres", [])) or None
    df_tmdb_bruto.at[idx_df, "duracao_min"] = detalhes.get("runtime")
    df_tmdb_bruto.at[idx_df, "orcamento"] = detalhes.get("budget")
    df_tmdb_bruto.at[idx_df, "receita"] = detalhes.get("revenue")
    df_tmdb_bruto.at[idx_df, "popularidade"] = detalhes.get("popularity")
    df_tmdb_bruto.at[idx_df, "media_votos"] = detalhes.get("vote_average")
    df_tmdb_bruto.at[idx_df, "contagem_votos"] = detalhes.get("vote_count")
    df_tmdb_bruto.at[idx_df, "keywords_tmdb"] = ", ".join(nomes_kw) or None
    df_tmdb_bruto.at[idx_df, "eh_adaptacao_livro"] = eh_adap
    recuperados += 1
    time.sleep(PAUSA)

print(f"\nFilmes recuperados: {recuperados}")
nova_taxa = df_tmdb_bruto['tmdb_id'].notna().mean() * 100
print(f"Nova taxa de match TMDB: {nova_taxa:.1f}%")

Reconsultando 64 filmes com título limpo...

[1/64] Crime Is Our Business (2008.0)
[2/64] The Adventures of Huck Finn (2012.0)
[3/64] Welcome Back Pinocchio (2007.0)
[4/64] Tom and Huck (2012.0)
[5/64] Helpless (2012.0)
[6/64] The Axe (2005.0)
[7/64] Battle Royale II: Requiem (2003.0)
[8/64] Don't Tell (2005.0)
[9/64] The Blue Room (2002.0)
[10/64] Behind the Sun (2001.0)
[11/64] The Karamazov Brothers (2008.0)
[12/64] Heading South (2005.0)
[13/64] The Crime of Padre Amaro (2002.0)
[15/64] Dangerous Liaisons (2012.0)
[16/64] Death Notice (2023.0)
[17/64] Nayakara Na Debadas (2019.0)
[18/64] Perfect Number (2012.0)
[19/64] Don Quixote, Knight Errant (2002.0)
[20/64] Honor of the Knights (2006.0)
[21/64] Dracula's Fiancee (2002.0)
[24/64] Nuptials of Dracula (2018.0)
[26/64] Morning Musume: Shinshun! Love Stories (2002.0)
[27/64] The Return of Buratino (2013.0)
[28/64] Buratino (2026.0)
[29/64] Fitoor (2016.0)
[30/64] Jajantaram Mamantaram (2003.0)
[32/64] A Chinese Tall Story (2005.0)


In [19]:
df_tmdb_bruto.to_csv(RAW / "tmdb_resultados.csv", index=False, encoding="utf-8")
salvar_proveniencia()
print("df_tmdb_bruto atualizado salvo (com os 48 recuperados).")

Proveniência salva em: dados_brutos/proveniencia.json (3280 entradas)
df_tmdb_bruto atualizado salvo (com os 48 recuperados).


## Limpeza, tratamento e integração das três fontes

Com os três brutos coletados e preservados (`dados_brutos/`), seguimos o fluxo de limpeza:
**profiling → chaves de integração → duplicatas → tipos → ausentes → outliers → join → validação →
features → salvar**. Nenhuma célula altera os DataFrames `*_bruto`: trabalhamos sobre cópias (`.copy()`),
preservando a fonte original em memória e em disco.

Há **duas chaves de
integração**, porque as fontes descrevem coisas diferentes:
- **Wikipédia × Google Books:** casadas pelo **livro** (título do livro + autor).
- **Wikipédia × TMDB:** casadas pelo **filme** (título do filme + ano).

A Wikipédia é o esqueleto: cada par livro→filme é uma linha, enriquecida pelos dados do livro
(Google Books) e do filme (TMDB).

### 1. Profiling inicial das três fontes brutas

Antes de qualquer decisão de limpeza, olhamos as dimensões, tipos e ausentes das três tabelas.

In [20]:
for nome, dframe in [
    ("Wikipédia (pares livro→filme)", df_wikipedia_bruto),
    ("Google Books (livros)", df_google_books_bruto),
    ("TMDB (filmes)", df_tmdb_bruto),
]:
    print("=" * 60)
    print(nome)
    print("=" * 60)
    print(f"Dimensões: {dframe.shape}")
    dframe.info()
    print()

Wikipédia (pares livro→filme)
Dimensões: (3821, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3821 entries, 0 to 3820
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   titulo_livro_wiki          3821 non-null   object 
 1   ano_publicacao_livro_wiki  3286 non-null   float64
 2   autor_wiki                 3286 non-null   object 
 3   titulo_filme_wiki          3818 non-null   object 
 4   ano_filme_wiki             3772 non-null   float64
 5   url_lista                  3821 non-null   object 
dtypes: float64(2), object(4)
memory usage: 179.2+ KB

Google Books (livros)
Dimensões: (522, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522 entries, 0 to 521
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   titulo_livro_wiki    522 non-null    object 
 1   titulo_google_books  485 non-null    ob

### 2. Chaves e limpeza de títulos

A Wikipédia é o esqueleto (911 pares livro→filme). As outras duas fontes entram assim:
- **TMDB** foi coletada linha a linha a partir do esqueleto, na mesma ordem — casamos **por posição**
  (exato, sem risco de erro de texto).
- **Google Books** tem 522 livros únicos para 911 linhas — casamos por **chave do livro**
  (título limpo + autor, normalizados), pois vários pares compartilham o mesmo livro.

Também limpamos trechos de tradução dos títulos de filme (`(French: ...)`, `(German: ...)`).

In [21]:
import unicodedata

def normalizar(texto):
    if pd.isna(texto):
        return None
    t = str(texto).strip().lower()
    t = unicodedata.normalize("NFKD", t)
    t = "".join(c for c in t if not unicodedata.combining(c))
    t = re.sub(r"[^a-z0-9\s]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t or None

def limpar_titulo_filme(titulo):
    if not isinstance(titulo, str):
        return titulo
    t = re.sub(r"\s*\((?:[A-Z][a-z]+:\s*)?[^)]*\)", "", titulo)
    return re.sub(r"\s+", " ", t).strip()

df_wikipedia_t = df_wikipedia.copy().reset_index(drop=True)
df_google_books = df_google_books_bruto.copy()

df_wikipedia_t["titulo_filme_limpo"] = df_wikipedia_t["titulo_filme_wiki"].apply(limpar_titulo_filme)

# Chave do livro na WIKIPÉDIA: usa titulo_livro_limpo (existe aqui)
df_wikipedia_t["chave_livro"] = (
    df_wikipedia_t["titulo_livro_limpo"].apply(normalizar).fillna("")
    + "|" + df_wikipedia_t["autor_wiki"].apply(normalizar).fillna("")
)

# Chave do livro no GOOGLE BOOKS: a coluna do título chama-se titulo_livro_wiki

col_titulo_gb = "titulo_livro_limpo" if "titulo_livro_limpo" in df_google_books.columns else "titulo_livro_wiki"
df_google_books["chave_livro"] = (
    df_google_books[col_titulo_gb].apply(normalizar).fillna("")
    + "|" + df_google_books["autor_wiki"].apply(normalizar).fillna("")
)

print("Chave do livro criada.")
print("Coluna de título usada no GB:", col_titulo_gb)
print("Exemplo (wiki):", df_wikipedia_t["chave_livro"].iloc[0])
print("Exemplo (gb):  ", df_google_books["chave_livro"].iloc[0])
print("Chaves únicas — wiki:", df_wikipedia_t["chave_livro"].nunique(),
      "| gb:", df_google_books["chave_livro"].nunique())

Chave do livro criada.
Coluna de título usada no GB: titulo_livro_wiki
Exemplo (wiki): the 25th hour|david benioff
Exemplo (gb):   the 25th hour|david benioff
Chaves únicas — wiki: 522 | gb: 522


### 3. Duplicatas

O Google Books tem um registro por livro único; garantimos que não haja chave de livro repetida antes
do join (mantendo a primeira ocorrência). Na Wikipédia, cada linha é um par livro→filme distinto —
livros repetidos (várias adaptações do mesmo livro) são legítimos e **não** são removidos.

In [22]:
dups_gb = df_google_books.duplicated(subset=["chave_livro"]).sum()
print(f"Duplicatas de chave_livro no Google Books: {dups_gb}")
df_google_books = df_google_books.drop_duplicates(subset=["chave_livro"], keep="first")
print(f"Livros únicos após dedup: {len(df_google_books)}")

Duplicatas de chave_livro no Google Books: 0
Livros únicos após dedup: 522


### 4. Integração (join) das três fontes

- **Wikipédia + TMDB:** por **posição** (índice), pois o `df_tmdb` foi construído linha a linha a
  partir do esqueleto — a linha *i* de um corresponde à linha *i* do outro.
- **Wikipédia + Google Books:** por **`chave_livro`** (`how="left"`), trazendo a avaliação do livro
  para todas as linhas daquele livro.

In [23]:
#Wikipédia + TMDB por posição (índice)
colunas_tmdb = [
    "tmdb_id", "titulo_tmdb", "data_lancamento", "idioma_original", "generos",
    "duracao_min", "orcamento", "receita", "popularidade", "media_votos",
    "contagem_votos", "keywords_tmdb", "eh_adaptacao_livro",
]
df_tmdb_idx = df_tmdb_bruto.reset_index(drop=True)[colunas_tmdb]

# confirmação de segurança: mesmos tamanhos e ordem
assert len(df_wikipedia_t) == len(df_tmdb_idx), "Wikipédia e TMDB com tamanhos diferentes!"

df_integrado = pd.concat([df_wikipedia_t.reset_index(drop=True), df_tmdb_idx], axis=1)

#  Wikipédia + Google Books por chave_livro
colunas_gb = [
    "chave_livro", "titulo_google_books", "autor_google_books",
    "nota_media", "n_avaliacoes", "categorias", "idioma", "editora",
]
df_integrado = df_integrado.merge(
    df_google_books[colunas_gb], on="chave_livro", how="left"
)

# Flag de disponibilidade da avaliação do livro
df_integrado["tem_avaliacoes_google_books"] = df_integrado["n_avaliacoes"].notna()
# Flag de match do filme
df_integrado["tem_match_tmdb"] = df_integrado["tmdb_id"].notna()

print("Dimensões da base integrada:", df_integrado.shape)
display(df_integrado.head())

Dimensões da base integrada: (911, 32)


,titulo_livro_wiki,ano_publicacao_livro_wiki,autor_wiki,titulo_filme_wiki,ano_filme_wiki,url_lista,titulo_livro_limpo,eh_serie,titulo_filme_limpo,chave_livro,...,eh_adaptacao_livro,titulo_google_books,autor_google_books,nota_media,n_avaliacoes,categorias,idioma,editora,tem_avaliacoes_google_books,tem_match_tmdb
0,The 25th Hour,2001.0,David Benioff,25th Hour,2002.0,https://en.wikipedia.org/wiki/List_of_fiction_...,The 25th Hour,False,25th Hour,the 25th hour|david benioff,...,True,The 25th Hour,David Benioff,4.0,1.0,Fiction,en,Plume Books,True,True
1,"3 Assassins (グラスホッパー, Gurasuhoppā)",2004.0,Kōtarō Isaka,Grasshopper,2015.0,https://en.wikipedia.org/wiki/List_of_fiction_...,3 Assassins,False,Grasshopper,3 assassins|kotaro isaka,...,True,Waltz,"Kotaro Isaka, Megumi Osuga",NaN,NaN,None,fr,None,False,True
2,4.50 from Paddington,1957.0,Agatha Christie,Crime Is Our Business (French: Le Crime est no...,2008.0,https://en.wikipedia.org/wiki/List_of_fiction_...,4.50 from Paddington,False,Crime Is Our Business,450 from paddington|agatha christie,...,True,4.50 from Paddington,Agatha Christie,NaN,NaN,Detective and mystery stories,en,None,False,True
3,"69 (シクスティナイン, Shikusutinain)",1987.0,Ryu Murakami,69,2004.0,https://en.wikipedia.org/wiki/List_of_fiction_...,69,False,69,69|ryu murakami,...,False,Sixty-Nine,Ryu Murakami,NaN,NaN,Fiction,en,Pushkin Press,False,True
4,À ton image,1998.0,Louise L. Lambrichs,À ton image,2004.0,https://en.wikipedia.org/wiki/List_of_fiction_...,À ton image,False,À ton image,a ton image|louise l lambrichs,...,False,A ton image,Louise L. Lambrichs,NaN,NaN,Fiction,fr,None,False,True


### 5. Correção de tipos de dados

Convertemos os campos numéricos que a TMDB/Google Books devolveram como texto, e derivamos o **ano de
lançamento do filme** a partir da data completa. Também limpamos resíduos de idioma nos títulos de
livro (ex.: caracteres japoneses entre parênteses) para a base final ficar legível.

In [24]:
# Data de lançamento do filme -> datetime, e ano derivado
df_integrado["data_lancamento"] = pd.to_datetime(df_integrado["data_lancamento"], errors="coerce")
df_integrado["ano_lancamento_filme"] = df_integrado["data_lancamento"].dt.year

# Campos numéricos da TMDB
for coluna in ["duracao_min", "orcamento", "receita", "popularidade",
               "media_votos", "contagem_votos"]:
    df_integrado[coluna] = pd.to_numeric(df_integrado[coluna], errors="coerce")

# Campos numéricos do Google Books
df_integrado["nota_media"] = pd.to_numeric(df_integrado["nota_media"], errors="coerce")
df_integrado["n_avaliacoes"] = pd.to_numeric(df_integrado["n_avaliacoes"], errors="coerce")

# Anos da Wikipédia como inteiros nullable
df_integrado["ano_publicacao_livro"] = pd.to_numeric(
    df_integrado["ano_publicacao_livro_wiki"], errors="coerce")

# Limpa resíduos de idioma no título do livro (ex.: "3 Assassins (グラスホッパー, ...)")
df_integrado["titulo_livro"] = df_integrado["titulo_livro_limpo"]

print("Tipos após correção:")
print(df_integrado[["ano_publicacao_livro", "ano_lancamento_filme", "nota_media",
                    "orcamento", "receita", "duracao_min"]].dtypes)

Tipos após correção:
ano_publicacao_livro    float64
ano_lancamento_filme    float64
nota_media              float64
orcamento               float64
receita                 float64
duracao_min             float64
dtype: object


In [25]:
# Correção de match: o fallback de busca sem ano às vezes casou com uma adaptação
# ANTIGA do mesmo livro (ex.: Dorian Gray 1945 em vez da versão recente). Quando o
# ano do filme na TMDB é muito anterior ao ano indicado pela Wikipédia, o match é de
# outra obra, invalidamos esses dados da TMDB para não contaminar a análise.
TOLERANCIA_ANOS = 5

divergencia = (
    df_integrado["ano_lancamento_filme"].notna()
    & df_integrado["ano_filme_wiki"].notna()
    & ((df_integrado["ano_filme_wiki"] - df_integrado["ano_lancamento_filme"]) > TOLERANCIA_ANOS)
)
print(f"Matches invalidados por divergência de ano (> {TOLERANCIA_ANOS} anos): {divergencia.sum()}")

# Anula os campos vindos da TMDB nessas linhas e rebaixa a flag
colunas_tmdb_anular = [
    "tmdb_id", "titulo_tmdb", "data_lancamento", "ano_lancamento_filme",
    "idioma_original", "generos", "duracao_min", "orcamento", "receita",
    "popularidade", "media_votos", "contagem_votos", "keywords_tmdb",
]
for col in colunas_tmdb_anular:
    if col in df_integrado.columns:
        df_integrado.loc[divergencia, col] = np.nan
df_integrado.loc[divergencia, "eh_adaptacao_livro"] = False
df_integrado["tem_match_tmdb"] = df_integrado["tmdb_id"].notna()

print(f"Nova taxa de match TMDB válido: {df_integrado['tem_match_tmdb'].mean()*100:.1f}%")

Matches invalidados por divergência de ano (> 5 anos): 16
Nova taxa de match TMDB válido: 96.5%


### 6. Dados ausentes

Decisões por coluna, não um `fillna` genérico:
- **`orcamento`/`receita`/`duracao_min` iguais a 0** na TMDB significam "não divulgado" → viram `NaN`,
  para não distorcer médias e outliers.
- **`nota_media`/`n_avaliacoes` ausentes:** o livro existe mas não tem avaliação no Google Books.
  Mantemos `NaN` (não inventamos nota) e a flag `tem_avaliacoes_google_books`.
- **`tmdb_id` ausente:** filme não encontrado; mantemos a linha e a flag `tem_match_tmdb`.
- **Categóricas textuais ausentes** (`generos`, `categorias`, `editora`, `idioma`): rótulo explícito
  `"Não informado"`.

In [26]:
# 0 = "não divulgado" na TMDB -> NaN
for coluna in ["orcamento", "receita", "duracao_min"]:
    df_integrado[coluna] = df_integrado[coluna].replace(0, np.nan)

# Categóricas textuais ausentes -> rótulo explícito
for coluna in ["generos", "categorias", "editora", "idioma", "idioma_original"]:
    if coluna in df_integrado.columns:
        df_integrado[coluna] = df_integrado[coluna].fillna("Não informado")

# eh_adaptacao_livro: para linhas sem match TMDB fica NaN; para as que casaram mas
# não tinham a keyword, é False. Normaliza para booleano (sem match -> False).
df_integrado["eh_adaptacao_livro"] = df_integrado["eh_adaptacao_livro"].fillna(False).astype(bool)

print("Resumo de ausentes após as decisões:")
nulos = df_integrado.isnull().sum()
print(nulos[nulos > 0].sort_values(ascending=False))

Resumo de ausentes após as decisões:
n_avaliacoes                 758
nota_media                   758
receita                      428
orcamento                    408
autor_wiki                   162
ano_publicacao_livro_wiki    162
ano_publicacao_livro         162
keywords_tmdb                142
autor_google_books            69
titulo_google_books           62
duracao_min                   42
data_lancamento               33
ano_lancamento_filme          33
tmdb_id                       32
media_votos                   32
popularidade                  32
titulo_tmdb                   32
contagem_votos                32
dtype: int64


/tmp/ipykernel_815/1384533827.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_integrado["eh_adaptacao_livro"] = df_integrado["eh_adaptacao_livro"].fillna(False).astype(bool)


### 7. Outliers (bilheteria, orçamento, avaliações)

Dados financeiros de cinema são naturalmente assimétricos: poucos blockbusters puxam a média. Isso
não é erro, é a distribuição real. Por isso **detectamos** outliers pelo método IQR, mas **não
aplicamos capping** aos valores financeiros, um blockbuster real é dado relevante para a pergunta
motivadora. A detecção é documentada para justificar a escolha.

In [27]:
def detectar_outliers_iqr(serie):
    dados = serie.dropna()
    q1, q3 = dados.quantile(0.25), dados.quantile(0.75)
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mascara = (serie < lim_inf) | (serie > lim_sup)
    return mascara.fillna(False), lim_inf, lim_sup

for coluna in ["receita", "orcamento", "popularidade", "n_avaliacoes"]:
    mascara, li, ls = detectar_outliers_iqr(df_integrado[coluna])
    print(f"{coluna:15s} | IQR [{li:,.1f}, {ls:,.1f}] | "
          f"outliers: {mascara.sum()} de {df_integrado[coluna].notna().sum()} válidos")

# Decisão registrada: não fazer capping nas variáveis financeiras/popularidade.

receita         | IQR [-206,204,829.0, 377,839,963.0] | outliers: 64 de 483 válidos
orcamento       | IQR [-75,000,000.0, 157,000,000.0] | outliers: 33 de 503 válidos
popularidade    | IQR [-8.6, 18.9] | outliers: 73 de 879 válidos
n_avaliacoes    | IQR [-6.5, 13.5] | outliers: 25 de 153 válidos


### 8. Engenharia de features

- **`retorno_financeiro`**: `(receita - orçamento) / orçamento`, quando o orçamento é conhecido e
  positivo — medida de desempenho comercial do filme. Depende só de campos da TMDB (mesma obra),
  sem o problema de datas de edição que afetava a versão anterior.

In [28]:
# Retorno financeiro: (receita - orçamento) / orçamento.
# Exige um piso de orçamento (US$ 100 mil) para evitar orçamentos implausíveis
# na TMDB (valores de poucos milhares, quase sempre erro de cadastro) que geram
# retornos absurdos. Também exige receita > 0 (senão o "retorno" não faz sentido).
ORCAMENTO_MINIMO = 100_000

orcamento_valido = (df_integrado["orcamento"] >= ORCAMENTO_MINIMO)
receita_valida = (df_integrado["receita"] > 0)

df_integrado["retorno_financeiro"] = np.where(
    orcamento_valido & receita_valida,
    (df_integrado["receita"] - df_integrado["orcamento"]) / df_integrado["orcamento"],
    np.nan,
)

print(f"Filmes com retorno calculável: {df_integrado['retorno_financeiro'].notna().sum()}")
display(df_integrado[["titulo_filme_limpo", "retorno_financeiro"]].describe())

# Mostra os retornos ainda muito altos, para você inspecionar se sobrou sujeira
print("\nMaiores retornos (conferir se são reais ou erro de orçamento):")
top = df_integrado.nlargest(10, "retorno_financeiro")[
    ["titulo_filme_limpo", "ano_lancamento_filme", "orcamento", "receita", "retorno_financeiro"]
]
display(top)

Filmes com retorno calculável: 428


,retorno_financeiro
count,428.000000
mean,2.203768
std,4.785192
min,-0.995994
25%,-0.108381
50%,1.123590
75%,2.858418
max,76.170440



Maiores retornos (conferir se são reais ou erro de orçamento):


,titulo_filme_limpo,ano_lancamento_filme,orcamento,receita,retorno_financeiro
880,Winnie-the-Pooh: Blood and Honey,2023.0,100000.0,7717044.0,76.170440
478,The Invisible Man,2020.0,7000000.0,144492724.0,19.641818
480,It Chapter One,2017.0,35000000.0,719766009.0,19.564743
881,Winnie-the-Pooh: Blood and Honey 2,2024.0,415000.0,7582541.0,17.271183
496,Monkey King: Hero Is Back,2015.0,10000000.0,153000000.0,14.300000
579,The Girl Who Played with Fire,2009.0,4400000.0,67126795.0,14.256090
791,The Best Exotic Marigold Hotel,2012.0,10000000.0,150501815.0,14.050182
336,Fifty Shades of Grey,2015.0,40000000.0,569651467.0,13.241287
827,The Twilight Saga: New Moon,2009.0,50000000.0,709827462.0,13.196549
196,"Crouching Tiger, Hidden Dragon",2000.0,17000000.0,213978518.0,11.586972


### 9. Seleção das colunas finais

Selecionamos e renomeamos as colunas para a base tratada ficar enxuta e legível. As colunas
intermediárias (títulos crus, chaves de integração) ficam preservadas apenas nos dados brutos.

In [29]:
# Mapa: coluna final <- coluna atual. Só o que interessa na base tratada.
colunas_finais = {
    # --- identificação da obra (Wikipédia) ---
    "titulo_livro":            "titulo_livro",
    "autor_wiki":              "autor",
    "ano_publicacao_livro":    "ano_publicacao_livro",
    "titulo_filme_limpo":      "titulo_filme",
    "ano_lancamento_filme":    "ano_lancamento_filme",
    "eh_serie":                "eh_serie",
    "url_lista":               "url_fonte",
    # --- livro (Google Books) ---
    "nota_media":              "nota_media_livro",
    "n_avaliacoes":            "n_avaliacoes_livro",
    "categorias":              "categorias_livro",
    "editora":                 "editora",
    "idioma":                  "idioma_livro",
    "tem_avaliacoes_google_books": "tem_avaliacoes_livro",
    # --- filme (TMDB) ---
    "tmdb_id":                 "tmdb_id",
    "data_lancamento":         "data_lancamento_filme",
    "idioma_original":         "idioma_filme",
    "generos":                 "generos_filme",
    "duracao_min":             "duracao_min",
    "orcamento":               "orcamento",
    "receita":                 "receita",
    "popularidade":            "popularidade",
    "media_votos":             "media_votos_filme",
    "contagem_votos":          "contagem_votos_filme",
    "tem_match_tmdb":          "tem_match_tmdb",
    "eh_adaptacao_livro":      "eh_adaptacao_livro",
    # --- feature derivada ---
    "retorno_financeiro":      "retorno_financeiro",
}

df_final = df_integrado[list(colunas_finais.keys())].rename(columns=colunas_finais)
print("Base final enxuta:", df_final.shape)
print("Colunas:", list(df_final.columns))
display(df_final.head())

Base final enxuta: (911, 26)
Colunas: ['titulo_livro', 'autor', 'ano_publicacao_livro', 'titulo_filme', 'ano_lancamento_filme', 'eh_serie', 'url_fonte', 'nota_media_livro', 'n_avaliacoes_livro', 'categorias_livro', 'editora', 'idioma_livro', 'tem_avaliacoes_livro', 'tmdb_id', 'data_lancamento_filme', 'idioma_filme', 'generos_filme', 'duracao_min', 'orcamento', 'receita', 'popularidade', 'media_votos_filme', 'contagem_votos_filme', 'tem_match_tmdb', 'eh_adaptacao_livro', 'retorno_financeiro']


,titulo_livro,autor,ano_publicacao_livro,titulo_filme,ano_lancamento_filme,eh_serie,url_fonte,nota_media_livro,n_avaliacoes_livro,categorias_livro,...,generos_filme,duracao_min,orcamento,receita,popularidade,media_votos_filme,contagem_votos_filme,tem_match_tmdb,eh_adaptacao_livro,retorno_financeiro
0,The 25th Hour,David Benioff,2001.0,25th Hour,2002.0,False,https://en.wikipedia.org/wiki/List_of_fiction_...,4.0,1.0,Fiction,...,"Crime, Drama",135.0,5000000.0,23932055.0,7.3098,7.312,2647.0,True,True,3.786411
1,3 Assassins,Kōtarō Isaka,2004.0,Grasshopper,2015.0,False,https://en.wikipedia.org/wiki/List_of_fiction_...,NaN,NaN,Não informado,...,"Crime, Thriller, Action, Drama",119.0,NaN,NaN,1.3858,5.900,15.0,True,True,NaN
2,4.50 from Paddington,Agatha Christie,1957.0,Crime Is Our Business,2008.0,False,https://en.wikipedia.org/wiki/List_of_fiction_...,NaN,NaN,Detective and mystery stories,...,"Comedy, Crime, Mystery",109.0,12000000.0,NaN,1.3664,5.888,98.0,True,True,NaN
3,69,Ryu Murakami,1987.0,69,2004.0,False,https://en.wikipedia.org/wiki/List_of_fiction_...,NaN,NaN,Fiction,...,"Drama, Comedy",114.0,NaN,NaN,2.2873,6.500,15.0,True,False,NaN
4,À ton image,Louise L. Lambrichs,1998.0,À ton image,2004.0,False,https://en.wikipedia.org/wiki/List_of_fiction_...,NaN,NaN,Fiction,...,"Drama, Science Fiction, Thriller",94.0,NaN,NaN,1.7590,4.833,18.0,True,False,NaN


### 9.1 Validação de sanidade da base final

Conferência automática de que os valores estão em faixas plausíveis antes de salvar.

In [30]:
def checar(condicao_ok, descricao):
    n = (~condicao_ok).sum()
    print(f"[{'OK ' if n == 0 else f'FALHOU ({n})'}] {descricao}")

d = df_final
print("Validação de faixas (linhas com valor presente):\n")
checar(d["nota_media_livro"].dropna().between(0, 5).reindex(d.index, fill_value=True),
       "nota_media_livro entre 0 e 5")
checar(d["media_votos_filme"].dropna().between(0, 10).reindex(d.index, fill_value=True),
       "media_votos_filme entre 0 e 10")
checar((d["receita"].fillna(0) >= 0), "receita não-negativa")
checar((d["orcamento"].fillna(0) >= 0), "orcamento não-negativo")
checar(d["ano_lancamento_filme"].dropna().between(1995, 2027).reindex(d.index, fill_value=True),
       "ano do filme (TMDB) plausível — recorte ~2000 com folga")
checar(d["ano_publicacao_livro"].dropna().between(1000, 2027).reindex(d.index, fill_value=True),
       "ano do livro plausível")
checar(d["duracao_min"].dropna().between(1, 600).reindex(d.index, fill_value=True),
       "duração entre 1 e 600 min")

print(f"\nDimensões finais: {d.shape}")
print(f"Taxa de match TMDB:        {d['tem_match_tmdb'].mean()*100:.1f}%")
print(f"Confirmados adaptação (818): {d['eh_adaptacao_livro'].mean()*100:.1f}%")
print(f"Com nota de livro:          {d['tem_avaliacoes_livro'].mean()*100:.1f}%")

Validação de faixas (linhas com valor presente):

[OK ] nota_media_livro entre 0 e 5
[OK ] media_votos_filme entre 0 e 10
[OK ] receita não-negativa
[OK ] orcamento não-negativo
[OK ] ano do filme (TMDB) plausível — recorte ~2000 com folga
[OK ] ano do livro plausível
[OK ] duração entre 1 e 600 min

Dimensões finais: (911, 26)
Taxa de match TMDB:        96.5%
Confirmados adaptação (818): 46.2%
Com nota de livro:          16.8%


In [31]:
# Quais linhas falharam na checagem de ano do filme?
fora = d[d["ano_lancamento_filme"].notna() & ~d["ano_lancamento_filme"].between(2000, 2027)]
print(f"Linhas com ano de filme fora de 2000-2027: {len(fora)}")
display(fora[["titulo_filme", "ano_lancamento_filme", "tem_match_tmdb"]])

Linhas com ano de filme fora de 2000-2027: 3


,titulo_filme,ano_lancamento_filme,tem_match_tmdb
259,The Mark of Dracula,1997.0,True
470,I Still Know What You Did Last Summer,1998.0,True
890,Woman Wanted,1999.0,True


### 9.2 Dicionário de variáveis (dataset card A.3)

Ficha de cada coluna da base tratada, exportada também em CSV para o dataset card.

In [32]:
dicionario = [
    ("titulo_livro",         "texto",             "Título do livro (Wikipédia, limpo)", "-"),
    ("autor",                "categórica",        "Autor(es) do livro (Wikipédia)", "-"),
    ("ano_publicacao_livro", "numérica discreta", "Ano da 1ª edição do livro (Wikipédia)", "ano"),
    ("titulo_filme",         "texto",             "Título do filme (Wikipédia, limpo)", "-"),
    ("ano_lancamento_filme", "numérica discreta", "Ano de lançamento do filme (TMDB)", "ano"),
    ("eh_serie",             "booleana",          "Se a obra é uma série/coletânea agrupada na fonte", "-"),
    ("url_fonte",            "texto",             "URL da lista da Wikipédia (atribuição)", "-"),
    ("nota_media_livro",     "numérica contínua", "Nota média do livro (Google Books)", "0–5"),
    ("n_avaliacoes_livro",   "numérica discreta", "Nº de avaliações do livro (Google Books)", "contagem"),
    ("categorias_livro",     "categórica",        "Categorias do livro (Google Books)", "-"),
    ("editora",              "categórica",        "Editora do livro (Google Books)", "-"),
    ("idioma_livro",         "categórica",        "Idioma da edição do livro (Google Books)", "ISO"),
    ("tem_avaliacoes_livro", "booleana",          "Se o livro tem avaliação no Google Books", "-"),
    ("tmdb_id",              "numérica discreta", "Identificador do filme na TMDB", "-"),
    ("data_lancamento_filme","data/hora",         "Data de lançamento do filme (TMDB)", "data"),
    ("idioma_filme",         "categórica",        "Idioma original do filme (TMDB)", "ISO"),
    ("generos_filme",        "categórica",        "Gêneros do filme (TMDB)", "-"),
    ("duracao_min",          "numérica contínua", "Duração do filme (0→NaN)", "minutos"),
    ("orcamento",            "numérica contínua", "Orçamento do filme (0→NaN)", "USD nominal"),
    ("receita",              "numérica contínua", "Receita/bilheteria do filme (0→NaN)", "USD nominal"),
    ("popularidade",         "numérica contínua", "Índice de popularidade (TMDB)", "score TMDB"),
    ("media_votos_filme",    "numérica contínua", "Nota média do filme (TMDB)", "0–10"),
    ("contagem_votos_filme", "numérica discreta", "Nº de votos do filme (TMDB)", "contagem"),
    ("tem_match_tmdb",       "booleana",          "Se o filme foi encontrado na TMDB", "-"),
    ("eh_adaptacao_livro",   "booleana",          "Se a TMDB confirma o filme como adaptação de livro (keyword 818)", "-"),
    ("retorno_financeiro",   "numérica contínua", "(receita - orçamento) / orçamento, se orçamento > 0", "razão"),
]
df_dicionario = pd.DataFrame(dicionario, columns=["Variável", "Tipo", "Descrição", "Unidade"])

faltando = set(df_final.columns) - set(df_dicionario["Variável"])
sobrando = set(df_dicionario["Variável"]) - set(df_final.columns)
if faltando: print("ATENÇÃO — colunas sem entrada no dicionário:", faltando)
if sobrando: print("ATENÇÃO — entradas que não existem na base:", sobrando)
if not faltando and not sobrando:
    print(f"Dicionário cobre exatamente as {len(df_final.columns)} colunas da base.")

df_dicionario.to_csv(TRATADO / "dicionario_variaveis.csv", index=False, encoding="utf-8")
display(df_dicionario)

Dicionário cobre exatamente as 26 colunas da base.


,Variável,Tipo,Descrição,Unidade
0,titulo_livro,texto,"Título do livro (Wikipédia, limpo)",-
1,autor,categórica,Autor(es) do livro (Wikipédia),-
2,ano_publicacao_livro,numérica discreta,Ano da 1ª edição do livro (Wikipédia),ano
3,titulo_filme,texto,"Título do filme (Wikipédia, limpo)",-
4,ano_lancamento_filme,numérica discreta,Ano de lançamento do filme (TMDB),ano
5,eh_serie,booleana,Se a obra é uma série/coletânea agrupada na fonte,-
6,url_fonte,texto,URL da lista da Wikipédia (atribuição),-
7,nota_media_livro,numérica contínua,Nota média do livro (Google Books),0–5
8,n_avaliacoes_livro,numérica discreta,Nº de avaliações do livro (Google Books),contagem
9,categorias_livro,categórica,Categorias do livro (Google Books),-


### 10. Salvamento da base tratada

Salvamos a base final em CSV e Parquet, separada dos brutos (que seguem intocados em `dados_brutos/`).

In [33]:
caminho_csv = TRATADO / "base_livros_filmes_tratada.csv"
df_final.to_csv(caminho_csv, index=False, encoding="utf-8")

caminho_parquet = TRATADO / "base_livros_filmes_tratada.parquet"
df_final.to_parquet(caminho_parquet, index=False)

salvar_proveniencia()
print(f"Salvo: {caminho_csv}")
print(f"Salvo: {caminho_parquet}")

# Prova de que os brutos seguem intocados
print("\nBrutos preservados:")
print(f"  Wikipédia (pares):    {df_wikipedia_bruto.shape}")
print(f"  Google Books (livros):{df_google_books_bruto.shape}")
print(f"  TMDB (filmes):        {df_tmdb_bruto.shape}")

Proveniência salva em: dados_brutos/proveniencia.json (3280 entradas)
Salvo: dados_tratados/base_livros_filmes_tratada.csv
Salvo: dados_tratados/base_livros_filmes_tratada.parquet

Brutos preservados:
  Wikipédia (pares):    (3821, 6)
  Google Books (livros):(522, 9)
  TMDB (filmes):        (911, 15)


In [34]:
# SEGURANÇA: remove a chave da API do Google Books dos dados de proveniência.

import json, re
from pathlib import Path

RAW = Path("dados_brutos")
prov_path = RAW / "proveniencia.json"

with open(prov_path, encoding="utf-8") as f:
    prov = json.load(f)

def mascarar_url(url):
    if not isinstance(url, str):
        return url
    # substitui o valor de key=... por REMOVIDA
    return re.sub(r"([?&]key=)[^&]+", r"\1REMOVIDA", url)

expostas = sum(1 for e in prov if isinstance(e.get("url"), str) and "key=" in e["url"])
print(f"Entradas com chave exposta na proveniência: {expostas}")

for e in prov:
    if "url" in e:
        e["url"] = mascarar_url(e["url"])
    # o parâmetro também pode estar em 'parametros'
    if isinstance(e.get("parametros"), dict) and "key" in e["parametros"]:
        e["parametros"]["key"] = "REMOVIDA"

with open(prov_path, "w", encoding="utf-8") as f:
    json.dump(prov, f, ensure_ascii=False, indent=2)

# confere que não sobrou nada
with open(prov_path, encoding="utf-8") as f:
    conteudo = f.read()
ainda_tem = re.search(r"key=(?!REMOVIDA)[A-Za-z0-9_\-]{10,}", conteudo)
print("Ainda há chave exposta na proveniência?", bool(ainda_tem))

# confere também os JSONs brutos do Google Books
suspeitos = []
for arq in (RAW / "google_books_json").glob("*.json"):
    txt = arq.read_text(encoding="utf-8")
    if re.search(r"AIza[A-Za-z0-9_\-]{20,}", txt):  # padrão de chave do Google
        suspeitos.append(arq.name)
print("JSONs brutos com padrão de chave Google:", suspeitos if suspeitos else "nenhum")

Entradas com chave exposta na proveniência: 522
Ainda há chave exposta na proveniência? False
JSONs brutos com padrão de chave Google: nenhum


In [35]:

# Inclui: dados brutos (com HTMLs, JSONs, proveniência), base tratada, e o dicionário.

import shutil

# zip da pasta de brutos (HTMLs, JSONs, CSVs crus, proveniencia.json)
shutil.make_archive("dados_brutos", "zip", "dados_brutos")
# zip da pasta de tratados (CSV, Parquet, dicionário)
shutil.make_archive("dados_tratados", "zip", "dados_tratados")

print("Gerados:")
print("  dados_brutos.zip")
print("  dados_tratados.zip")

# Confere o que ficou dentro de cada um
import zipfile
for nome in ["dados_brutos.zip", "dados_tratados.zip"]:
    with zipfile.ZipFile(nome) as z:
        arquivos = z.namelist()
    print(f"\n{nome} — {len(arquivos)} arquivos:")
    for a in arquivos[:15]:
        print("   ", a)
    if len(arquivos) > 15:
        print(f"    ... e mais {len(arquivos)-15}")

Gerados:
  dados_brutos.zip
  dados_tratados.zip

dados_brutos.zip — 3285 arquivos:
    google_books_json/
    tmdb_json/
    wikipedia_html/
    proveniencia.json
    wikipedia_listas_pares.csv
    tmdb_resultados.csv
    wikipedia_html/lista_pagina_04.html
    wikipedia_html/lista_pagina_02.html
    wikipedia_html/lista_pagina_01.html
    wikipedia_html/lista_pagina_03.html
    tmdb_json/tmdb_busca_0632.json
    tmdb_json/tmdb_detalhes_0797.json
    tmdb_json/tmdb_keywords_0339.json
    tmdb_json/tmdb_keywords_5036.json
    tmdb_json/tmdb_keywords_0514.json
    ... e mais 3270

dados_tratados.zip — 3 arquivos:
    base_livros_filmes_tratada.parquet
    base_livros_filmes_tratada.csv
    dicionario_variaveis.csv
